# AnalystLab Africa - Dataset Cleaning & Exploratory Data Analysis (EDA)
## E-commerce Transactions Dataset

In [190]:
#Importing the pandas library
import numpy as np

#Printing the outcome
print('Pandas successfully loaded')

import os

Pandas successfully loaded


In [191]:
#Loading the dataset and creating a copy to use instead of the original
df_raw = pd.read_csv("OnlineRetail.csv", encoding='ISO-8859-1')

df = df_raw.copy()

#Print upon successful completion
print('E-commerce dataset has been loaded')

E-commerce dataset has been loaded


In [192]:
#Displaying the first few rows
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [193]:
#Displaying the number of rows and columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [194]:
#Displaying the datatypes of each column
df.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

In [195]:
#Displaying Numerical Features
df.select_dtypes(include='number').columns

Index(['Quantity', 'UnitPrice', 'CustomerID'], dtype='object')

In [196]:
#Displaying Categorical Features
df.select_dtypes(include='object').columns

Index(['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate', 'Country'], dtype='object')

In [197]:
#Assessing possible unique identifiers
for col in df.columns:
    if df[col].nunique() == len(df):
        print(col)

df.duplicated(subset=['InvoiceNo', 'StockCode']).sum()

np.int64(10684)

## Description of the Dataset
The dataset contains online e-commerce data transactions extracted from a real-world dataset. 
    
It contains 541909 rows and 8 columns, namely, details such as the invoice number, stock code, description, quantity, invoice date, 
unit price, customer id and country.
    
From those 8 columns, there is only 1 integer column, the quantity column, which makes sense as you cannot purchase 1.5 or 0.5 of a product. 
This is also one of the numerical features.
    
There are 2 float columns, which represent the other numerical features (UnitPrice and CustomerID), 
and the remaining are considered categorical features but the true categorical feature would be 'Country'.
CustomerID can't really be considered a float as you can only have a solid number for customer ID, suggesting further future cleaning or assessment
must be considered when assessing the column.
    
The 'InvoiceDate', 'InvoiceNo', 'StockCode' columns would need to be converted to the correct formats upon cleaning.
Description is free text and would not need converting.

Although CustomerID and InvoiceNo would typically serve as primary keys in a normalised relational schema (e.g., separate Customers and Invoices tables), in this flat transactional dataset they function as foreign keys, since each value repeats across multiple rows. No column in this dataset, as structured, qualifies as a true unique identifier on its own.

## Missing Values

In [198]:
#Identifying columns with missing values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Description and CustomerID have missing values. 1454 and 135080 respectively

In [199]:
#The percentage of null values in relation to the dataset
(df.isnull().sum() / len(df) * 100).round(2)

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

Description is 0.27% and CustomerID is 24.93%

Since description is only 0.27% we can drop the null values

In [200]:
#Dropping the null values in description
df = df.dropna(subset=['Description'])

In [201]:
# We cannot perform customer-level analysis with the missing customer IDs.
# Adding demo customer IDs may be inaccurate as it can merge or split the same customer transactions
# We use a dataset that excludes the rows with the missing customerID
df_customers = df.dropna(subset=['CustomerID'])

In [202]:
# New dataset without the null Description rows
print(df.shape)  

#New dataset without the null CustomerID values
print(df_customers.shape)

#Number of rows with null CustomerIDs
print(df_customers['CustomerID'].isnull().sum())

(540455, 8)
(406829, 8)
0


## Duplicated Values

In [203]:
# Identifying duplicated values
df.duplicated().sum()

np.int64(5268)

In [204]:
#5268 duplicated rows
#Now to drop duplicated rows
df = df.drop_duplicates()
df.shape

(535187, 8)

In [205]:
#5268 duplicated rows were removed, leaveing the ne dataframe with 535187 rows

## Standardisation

In [206]:
#Fixing the dates
#Invoice date is the only column that has a date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.dtypes   #To show corrected column

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

In [207]:
#Checking for columns that need fixing
#InvoiceNo and StockCode are text objects but would be considered different from the other 2, Description and Country
#We have to get rid of the unnessessary spaces in the Description column and fix the casing of the descriptions
df['Description'] = df['Description'].str.strip()
df['Description'] = df['Description'].str.upper()
df['Description'].unique()[:20] #checking for the outcome

array(['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN',
       'CREAM CUPID HEARTS COAT HANGER',
       'KNITTED UNION FLAG HOT WATER BOTTLE',
       'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES',
       'GLASS STAR FROSTED T-LIGHT HOLDER', 'HAND WARMER UNION JACK',
       'HAND WARMER RED POLKA DOT', 'ASSORTED COLOUR BIRD ORNAMENT',
       "POPPY'S PLAYHOUSE BEDROOM", "POPPY'S PLAYHOUSE KITCHEN",
       'FELTCRAFT PRINCESS CHARLOTTE DOLL', 'IVORY KNITTED MUG COSY',
       'BOX OF 6 ASSORTED COLOUR TEASPOONS',
       'BOX OF VINTAGE JIGSAW BLOCKS', 'BOX OF VINTAGE ALPHABET BLOCKS',
       'HOME BUILDING BLOCK WORD', 'LOVE BUILDING BLOCK WORD',
       'RECIPE BOX WITH METAL HEART'], dtype=object)

In [208]:
#Fixing the casing of the Country column
df['Country'].unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Bahrain', 'Greece', 'Hong Kong', 'Singapore',
       'Lebanon', 'United Arab Emirates', 'Saudi Arabia',
       'Czech Republic', 'Canada', 'Unspecified', 'Brazil', 'USA',
       'European Community', 'Malta', 'RSA'], dtype=object)

In [209]:
# Checking for consistency in column names
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [210]:
# No action needed

In [211]:
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

## Data validation

In [212]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [213]:
#Validating the Quantity column
(df['Quantity'] < 0).sum()

np.int64(9725)

In [214]:
#Seeing the title of the InvoiceNo columns that start with a C. (Quantities that have a negative value)
df[df['Quantity'] < 0]['InvoiceNo'].head(10)

141    C536379
154    C536383
235    C536391
236    C536391
237    C536391
238    C536391
239    C536391
240    C536391
241    C536391
939    C536506
Name: InvoiceNo, dtype: object

In [215]:
df[df['Quantity'] < 0]['InvoiceNo'].astype(str).str.startswith('C').sum()

np.int64(9251)

In [216]:
df[(df['Quantity'] < 0) & (~df['InvoiceNo'].astype(str).str.startswith('C'))].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
7313,537032,21275,?,-30,2010-12-03 16:50:00,0.0,NaN,United Kingdom
13217,537425,84968F,CHECK,-20,2010-12-06 15:35:00,0.0,NaN,United Kingdom
13218,537426,84968E,CHECK,-35,2010-12-06 15:36:00,0.0,NaN,United Kingdom
13264,537432,35833G,DAMAGES,-43,2010-12-06 16:10:00,0.0,NaN,United Kingdom
21338,538072,22423,FAULTY,-13,2010-12-09 14:10:00,0.0,NaN,United Kingdom
21518,538090,20956,?,-723,2010-12-09 14:48:00,0.0,NaN,United Kingdom
22296,538161,46000S,DOTCOM SALES,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
22297,538162,46000M,DOTCOM SALES,-100,2010-12-09 17:25:00,0.0,NaN,United Kingdom
42564,540010,22501,REVERSE 21/5/10 ADJUSTMENT,-100,2011-01-04 11:13:00,0.0,NaN,United Kingdom
42566,540012,22502,REVERSE 21/5/10 ADJUSTMENT,-100,2011-01-04 11:14:00,0.0,NaN,United Kingdom


In [217]:
df = df[~((df['Quantity'] < 0) & (~df['InvoiceNo'].astype(str).str.startswith('C')))]

In [218]:
# Found that 474 of the 9,725 negative-quantity rows do NOT have a 
# cancellation invoice (InvoiceNo starting with "C"). On inspection, 
# these rows have UnitPrice = 0 and no CustomerID, with descriptions 
# like "DAMAGES", "FAULTY", "CHECK", and "REVERSE ADJUSTMENT" - 
# indicating internal stock corrections, not real customer transactions.
# These are removed since they carry no transaction value and would 
# distort sales/revenue analysis. The remaining 9,251 negative-quantity 
# rows (legitimate cancellations) are retained.

In [219]:
df.shape

(534713, 8)

In [220]:
(df['UnitPrice'] < 0).sum()

np.int64(2)

In [221]:
(df['UnitPrice'] == 0).sum()

np.int64(582)

In [222]:
df[df['UnitPrice'] < 0]
df[df['UnitPrice'] == 0].head(15) 

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
6391,536941,22734,AMAZON,20,2010-12-03 12:08:00,0.0,NaN,United Kingdom
6392,536942,22139,AMAZON,15,2010-12-03 12:08:00,0.0,NaN,United Kingdom
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
14335,537534,85064,CREAM SWEETHEART LETTER RACK,1,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14336,537534,84832,ZINC WILLIE WINKIE CANDLE STICK,1,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14337,537534,84692,BOX OF 24 COCKTAIL PARASOLS,2,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14338,537534,48184,DOORMAT ENGLISH ROSE,3,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14339,537534,48111,DOORMAT 3 SMILEY CATS,1,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14340,537534,22697,GREEN REGENCY TEACUP AND SAUCER,1,2010-12-07 11:48:00,0.0,NaN,United Kingdom
14341,537534,22682,FRENCH BLUE METAL DOOR SIGN 7,1,2010-12-07 11:48:00,0.0,NaN,United Kingdom


In [223]:
df[df['UnitPrice'] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,ADJUST BAD DEBT,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,ADJUST BAD DEBT,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [224]:
df['IsFreeItem'] = df['UnitPrice'] == 0

In [225]:
# Removing 2 rows that represent internal "bad debt" accounting adjustments,
# not real customer transactions - identified by non-standard InvoiceNo ("A" prefix),
# placeholder StockCode ("B"), and description "ADJUST BAD DEBT"
df = df[df['UnitPrice'] >= 0]

In [226]:
df.shape

(534711, 9)

In [227]:
# "EIRE" is the old/formal name for Ireland used in this dataset.
# Renamed to "Ireland" for consistency with modern country naming.
df['Country'] = df['Country'].replace('EIRE', 'Ireland')

In [228]:
'EIRE' in df['Country'].unique()

False

In [229]:
#Cheking for outliers
df['Quantity'].describe()
df['UnitPrice'].describe()

count    534711.000000
mean          4.690753
std          95.027557
min           0.000000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

In [230]:
df.sort_values('UnitPrice', ascending=False).head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsFreeItem
222681,C556445,M,MANUAL,-1,2011-06-10 15:31:00,38970.00,15098.0,United Kingdom,False
524602,C580605,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:36:00,17836.46,NaN,United Kingdom,False
43702,C540117,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:55:00,16888.02,NaN,United Kingdom,False
43703,C540118,AMAZONFEE,AMAZON FEE,-1,2011-01-05 09:57:00,16453.71,NaN,United Kingdom,False
15016,C537630,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:04:00,13541.33,NaN,United Kingdom,False
15017,537632,AMAZONFEE,AMAZON FEE,1,2010-12-07 15:08:00,13541.33,NaN,United Kingdom,False
16356,C537651,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:49:00,13541.33,NaN,United Kingdom,False
16232,C537644,AMAZONFEE,AMAZON FEE,-1,2010-12-07 15:34:00,13474.79,NaN,United Kingdom,False
524601,C580604,AMAZONFEE,AMAZON FEE,-1,2011-12-05 11:35:00,11586.50,NaN,United Kingdom,False
299982,A563185,B,ADJUST BAD DEBT,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,False


In [231]:
# These StockCodes represent non-product entries: "M" (manual entries), 
# "AMAZONFEE" (marketplace commission charges), and "B" (bad debt adjustments) - 
# not real customer purchases. Removing them since they would distort 
# revenue and price-based analysis.
df = df[~df['StockCode'].isin(['M', 'AMAZONFEE', 'B'])]
df.shape

(534110, 9)

In [232]:
df['UnitPrice'].describe()

count    534110.000000
mean          3.807552
std          24.038786
min           0.000000
25%           1.250000
50%           2.080000
75%           4.130000
max        8142.750000
Name: UnitPrice, dtype: float64

In [233]:
df.sort_values('UnitPrice', ascending=False).head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsFreeItem
173277,C551685,POST,POSTAGE,-1,2011-05-03 12:51:00,8142.75,16029.0,United Kingdom,False
173382,551697,POST,POSTAGE,1,2011-05-03 13:46:00,8142.75,16029.0,United Kingdom,False
297723,562955,DOT,DOTCOM POSTAGE,1,2011-08-11 10:14:00,4505.17,NaN,United Kingdom,False
493020,578149,DOT,DOTCOM POSTAGE,1,2011-11-23 11:11:00,2275.54,NaN,United Kingdom,False
524892,580610,DOT,DOTCOM POSTAGE,1,2011-12-05 11:48:00,2196.67,NaN,United Kingdom,False
525134,580612,DOT,DOTCOM POSTAGE,1,2011-12-05 11:58:00,2114.00,NaN,United Kingdom,False
502058,578833,DOT,DOTCOM POSTAGE,1,2011-11-25 15:23:00,2028.25,NaN,United Kingdom,False
431348,573585,DOT,DOTCOM POSTAGE,1,2011-10-31 14:41:00,2019.05,NaN,United Kingdom,False
150591,C549452,D,DISCOUNT,-1,2011-04-08 14:17:00,1867.86,17940.0,United Kingdom,False
533491,581023,DOT,DOTCOM POSTAGE,1,2011-12-07 10:35:00,1861.46,NaN,United Kingdom,False


In [234]:
# StockCodes "POST" (postage), "DOT" (dotcom postage), and "D" (discount) 
# represent shipping charges and discount adjustments, not real product sales.
# Removing them for the same reason as M/AMAZONFEE/B - they would distort 
# product-level and revenue analysis.
df = df[~df['StockCode'].isin(['POST', 'DOT', 'D'])]
df.shape

(532072, 9)

In [235]:
df['UnitPrice'].describe()

count    532072.000000
mean          3.336822
std           6.394786
min           0.000000
25%           1.250000
50%           2.080000
75%           4.130000
max        1100.440000
Name: UnitPrice, dtype: float64

In [236]:
df['Quantity'].describe()

count    532072.000000
mean         10.020259
std         217.690846
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [237]:
df[df['Quantity'].abs() > 5000]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsFreeItem
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom,False
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom,False
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom,False
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom,True
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,False
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom,False


In [238]:
# Checked extreme Quantity values (>5000 or <-5000 absolute). 
# These are legitimate wholesale transactions with real StockCodes/Descriptions,
# often paired with a matching cancellation shortly after (e.g. large order 
# placed then cancelled same day). Retained as valid, non-erroneous outliers.

In [239]:
# "Unspecified" and "European Community" left unchanged in the Country column.
# "Unspecified" represents genuinely unknown origin - no reliable way to correct it.
# "European Community" is a regional/customs designation, not a single country -
# reassigning it to one specific country would be guessing, not correcting.
# Both are retained as-is and noted as a known limitation of this dataset.

In [240]:
# Refreshing df_customers from the fully cleaned df
df_customers = df.dropna(subset=['CustomerID'])
df_customers.shape

(399855, 9)

In [241]:
df.to_csv('OnlineRetail_Cleaned.csv', index=False)

## Data Cleaning Summary

| Issue Found | Action Taken |
|---|---|
| Missing Description (0.27%, 1,454 rows) | Dropped; negligible, unusable for product analysis |
| Missing CustomerID (24.93%, 135,080 rows) | Retained in df for product/revenue-level analysis; separate df_customers subset created (399,855 rows) for customer-level analysis |
| Duplicate rows (5,268 full-row duplicates) | Removed via drop_duplicates() |
| InvoiceDate stored as text | Converted to proper datetime format |
| Description casing/whitespace inconsistency | Standardized: stripped whitespace, converted to uppercase |
| Column names | Reviewed; already consistent (CamelCase, no spaces/symbols); no changes needed |
| Negative Quantity (9,725 rows) | 9,251 legitimate cancellations (InvoiceNo starts with "C") retained; 474 internal stock adjustments (DAMAGES, FAULTY, CHECK, REVERSE ADJUSTMENT, £0 price, no CustomerID) removed |
| Non-product StockCodes with extreme UnitPrice (M, AMAZONFEE, B, POST, DOT, D) | Removed; internal fees, postage, discounts, and bad debt adjustments, not real product sales |
| Zero UnitPrice (582 rows) | Retained and flagged with new IsFreeItem column; legitimate free/promotional items |
| Extreme Quantity outliers (over 5,000 or under -5,000) | Investigated; confirmed legitimate wholesale orders (often paired with matching cancellations); retained as valid |
| Country: "EIRE" | Renamed to "Ireland" for consistency with modern naming |
| Country: "Unspecified" / "European Community" | Left unchanged; documented as known dataset limitation (no reliable way to correct without guessing) |

Final cleaned dataset: 532,072 rows x 9 columns (from an original 541,909 rows x 8 columns)

In [242]:
import os
print(os.getcwd())

C:\Users\HP


In [243]:
os.path.exists('OnlineRetail_Cleaned.csv')

True